In [201]:
import requests
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from fredapi import Fred
from datetime import datetime


## Sklearn
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

### Implementing the Principal Component Analysis (PCA) approach of Unsupervised learning from both First principle and using the Sklearn library

#### PCA From first principle

##### Extract yield of US treasuries from FRED api

In [5]:
fred = Fred(api_key='7d6f82145d680560385d9290efaed81f')

In [141]:
series_ids = {
    "1-Month": "DGS1MO",
    "3-Month": "DGS3MO",
    "6-Month": "DGS6MO",
    "1-Year": "DGS1",
    "2-Year": "DGS2",
    "3-Year": "DGS3",
    "5-Year": "DGS5",
    "7-year": "DGS7",
    "10-Year": "DGS10",
    "20-Year": "DGS20",
    "30-Year": "DGS30"
}

start_date = datetime.strptime("2016-02-16", "%Y-%m-%d").strftime("%Y-%m-%d")
end_date = datetime.strptime("2017-12-29", "%Y-%m-%d").strftime("%Y-%m-%d")

yield_data = {}
for label, series in series_ids.items():
    try:
        yield_data[label]=fred.get_series(series, observation_start=start_date, observation_end=end_date)
    except Exception as e:
        print(f"Error fetching {label}: {e}")

In [142]:
tyield_df = pd.DataFrame(yield_data)
tyield_df.index = pd.to_datetime(tyield_df.index)
tyield_df.dropna(axis=0, how='any', inplace=True)
#tyield_df = tyield_df/100 # Convert to percentage
tyield_df.tail(7)

,1-Month,3-Month,6-Month,1-Year,2-Year,3-Year,5-Year,7-year,10-Year,20-Year,30-Year
2017-12-20,1.22,1.38,1.51,1.72,1.87,1.98,2.24,2.40,2.49,2.71,2.88
2017-12-21,1.21,1.35,1.54,1.73,1.89,2.01,2.26,2.39,2.48,2.68,2.84
2017-12-22,1.15,1.33,1.54,1.73,1.91,2.01,2.26,2.40,2.48,2.68,2.83
2017-12-26,1.24,1.47,1.52,1.75,1.92,2.02,2.25,2.38,2.47,2.66,2.82
2017-12-27,1.18,1.44,1.53,1.75,1.89,1.99,2.22,2.34,2.42,2.59,2.75
2017-12-28,1.19,1.39,1.54,1.76,1.91,2.00,2.23,2.36,2.43,2.60,2.75
2017-12-29,1.28,1.39,1.53,1.76,1.89,1.98,2.20,2.33,2.40,2.58,2.74


##### Plot yields

In [129]:
fig = go.Figure()

maturities_to_plot = ['3-Month', '2-Year', '5-Year', '10-Year']
#maturities_to_plot = tyield_df.columns

for mat in maturities_to_plot:
    fig.add_trace(
        go.Scatter(
            x=tyield_df.index,
            y=tyield_df[mat],
            mode='markers',
            text=[f"{p:.2f}%" for p in tyield_df[mat]],
            name=mat,
        )
    )
    
    fig.update_layout(
        title='US Treasury Interest Rates across time for selected maturities',
        xaxis={'title': 'Days'},
        yaxis={'title': 'yields', 'ticksuffix': '%'},
        legend={'title': 'Maturities'},
        template='plotly_white',
        hovermode='x unified',
        width=1000,
        height=700,
    )

fig.show()

In [40]:
#### Starting PCA from scratch

In [224]:
#### Extract yields to numpy arrays
yields_np = np.array(tyield_df.values)
yields_np.shape

(471, 11)

In [225]:
##### Calculate covariance matrix
X_centred = yields_np - np.mean(yields_np, axis=0)
cov_matrix_yields = np.cov(yields_np, rowvar=False) ##row as variable is set to false because it is each column that is a variable

In [226]:
#### Extract eigenvalues and eigenvectors from data
eigenvalues_yields, eigenvectors_yields = np.linalg.eig(cov_matrix_yields)
print(f"Eigenvalues for the cov_matrix of yields with shape {cov_matrix_yields.shape} is : {eigenvalues_yields}")

Eigenvalues for the cov_matrix of yields with shape (11, 11) is : [1.08050149e+00 1.52961074e-01 9.99709334e-03 3.04073598e-03
 1.11737040e-03 4.09777910e-04 2.98984867e-04 2.21113513e-04
 9.58646225e-05 4.14014614e-05 6.16161003e-05]


In [227]:
### Sort eigenvalues in descending order
idx = np.argsort(eigenvalues_yields)[::-1]
eigenvalues_yields = eigenvalues_yields[idx]
eigenvectors_yields = eigenvectors_yields[:, idx]

# eigenvectors_yields_as_is = eigenvectors_yields.copy()
# eigenvectors_yields[:, 0] *= -1
eigenvectors_yields

array([[-0.29958527, -0.3291228 ,  0.36985553, -0.67025758, -0.44008084,
         0.10810461,  0.00499173, -0.0792789 ,  0.05870996,  0.01374883,
         0.02418263],
       [-0.3228561 , -0.34649352,  0.31213973,  0.02061964,  0.54384733,
        -0.60160024, -0.08980895,  0.09175763,  0.0045017 , -0.05946077,
         0.01077372],
       [-0.31669174, -0.34262539,  0.16321315,  0.33002611,  0.25967687,
         0.61639225,  0.40732469, -0.07405464, -0.14295613,  0.08672707,
         0.00560353],
       [-0.31654365, -0.29565086, -0.14884894,  0.42506756, -0.37226904,
         0.08327353, -0.53365142,  0.38792622,  0.16374258, -0.02883021,
        -0.03346838],
       [-0.31486121, -0.10775904, -0.39034929,  0.13545132, -0.29677271,
        -0.2956162 ,  0.19680613, -0.42833688, -0.42658464, -0.36482635,
         0.07609169],
       [-0.32761217,  0.03579479, -0.38337191, -0.03567051, -0.0011002 ,
        -0.16700734,  0.14241503, -0.20818432,  0.35284069,  0.7243476 ,
         0.049

In [228]:
#### Calculating the explanatory power of each Principal Component (PC) as derived from the eigenvalues
relative_explanation = eigenvalues_yields/eigenvalues_yields.sum()
percentages = relative_explanation * 100
labels = [f"eigV_PC{i+1}" for i in range(len(eigenvalues_yields))]

# Option 1: Print with formatting
for i, pct in enumerate(percentages):
    print(f"PC_{i+1}: {pct:.2f}%")

PC_1: 86.53%
PC_2: 12.25%
PC_3: 0.80%
PC_4: 0.24%
PC_5: 0.09%
PC_6: 0.03%
PC_7: 0.02%
PC_8: 0.02%
PC_9: 0.01%
PC_10: 0.00%
PC_11: 0.00%


In [229]:
#### Plotting the relative values of the Eigenvalues

fig_pct = go.Figure(
    data = go.Bar(
        x = labels,
        y = percentages,
        #text=[f"{p:.2f}%" for p in percentages],
        textposition='outside',
        marker=dict(color='teal')
    )
)

fig_pct.update_layout(
    title='Relative Weights as Percentages',
    xaxis_title='Variables',
    yaxis_title='Percentages (%)',
    yaxis=dict(ticksuffix='%'),
    template='plotly_white',
    width=1000,
    height=700,
)

fig_pct.show()

In [230]:
#### From the above plot it is obvious that it is the first two PCs that really matters

print(f"PC_1 explains {percentages[0]:.2f}% of variance, whereas PC_1 and PC_2 explains {(relative_explanation[:2]).sum()*100:.2f}% of variance, while PC_1, PC_2 and PC_3 explains {(relative_explanation[:3]).sum()*100:.2f}% of variance")

PC_1 explains 86.53% of variance, whereas PC_1 and PC_2 explains 98.78% of variance, while PC_1, PC_2 and PC_3 explains 99.58% of variance


In [241]:
#### Selecting three PCs, i.t PC1 to PC3
k = 3 #number of PCs selected
selected_eigenvectors = eigenvectors_yields[:, :k]
selected_eigenvectors[:, 0] *= -1
print(f"PC1 to PC3 eigenvector loadings are: ")
selected_eigenvectors

PC1 to PC3 eigenvector loadings are: 


array([[ 0.29958527, -0.3291228 ,  0.36985553],
       [ 0.3228561 , -0.34649352,  0.31213973],
       [ 0.31669174, -0.34262539,  0.16321315],
       [ 0.31654365, -0.29565086, -0.14884894],
       [ 0.31486121, -0.10775904, -0.39034929],
       [ 0.32761217,  0.03579479, -0.38337191],
       [ 0.333928  ,  0.19573225, -0.30014588],
       [ 0.31055227,  0.29297731, -0.16020542],
       [ 0.29397795,  0.33932703,  0.03791679],
       [ 0.26134447,  0.38801394,  0.3429931 ],
       [ 0.190728  ,  0.40506924,  0.42755031]])

In [242]:
selected_eigenvectors_df = pd.DataFrame(selected_eigenvectors, columns=['PC1', 'PC_2', 'PC3'], index=tyield_df.columns)
selected_eigenvectors_df

,PC1,PC_2,PC3
1-Month,0.299585,-0.329123,0.369856
3-Month,0.322856,-0.346494,0.312140
6-Month,0.316692,-0.342625,0.163213
1-Year,0.316544,-0.295651,-0.148849
2-Year,0.314861,-0.107759,-0.390349
3-Year,0.327612,0.035795,-0.383372
5-Year,0.333928,0.195732,-0.300146
7-year,0.310552,0.292977,-0.160205
10-Year,0.293978,0.339327,0.037917
20-Year,0.261344,0.388014,0.342993


In [243]:
fig_pcs = go.Figure()

for pc in selected_eigenvectors_df.columns:
    fig_pcs.add_trace(
        go.Scatter(
            x=selected_eigenvectors_df.index,
            y=selected_eigenvectors_df[pc],
            mode='lines + markers',
            line={'dash':'dash'},
            name=pc,
        )
    )
    
    fig_pcs.update_layout(
        title='Eigenvector Loadings for selected eigenvectors',
        xaxis={'title': 'Maturities'},
        yaxis={'title': 'Loadings'},
        legend={'title': 'Maturities'},
        template='plotly_white',
        width=1000,
        height=700,
    )

fig_pcs.show()

#### A possible economic interpretation of the eigenvectors:

**PC1**: if Interest Rates (IR) goes up, then PC1 goes up, viz-a-viz __ (Level/parallel shift) indicates/economic interpretation Duration, general macro sentiment (growth, inflation expectations, central bank stance) often explains 80-90% of variance<br>
**PC2**: for short maturities: if IR goes up, then PC2 goes down, 
<br> **PC2**    for long maturities: if IR goes up, then PC2 goes up ___ PC2 indicates Twist (slope of yield curve): measures steepening or flattening of the curve often explains 5-10% of variance <br>
**PC3**: for short and long maturities: if IT goes up, then PC3 goes up
<br> **PC3**      for mid-term maturities: if IR goes up then PC3 goes down (buterfly effect) ___PC3 indicating Curvature often explains 2-5% of variance


##### Factor Scores overtime

In [244]:
#### Factor scores
factor_scores = X_centred @ selected_eigenvectors_df
factor_scores

,PC1,PC_2,PC3
0,-1.148031,0.083041,0.111744
1,-1.065046,0.121336,0.141201
2,-1.167245,0.016736,0.148026
3,-1.124008,0.023188,0.098259
4,-1.081750,0.018176,0.100763
...,...,...,...
466,1.913976,-0.446665,-0.276386
467,1.972936,-0.547597,-0.218687
468,1.860384,-0.608795,-0.265820
469,1.878246,-0.587851,-0.291622


In [245]:
factor_scores_df = pd.DataFrame(factor_scores)
factor_scores_df.columns = ['PC1', 'PC2', 'PC3']
factor_scores_df

,PC1,PC2,PC3
0,-1.148031,0.083041,0.111744
1,-1.065046,0.121336,0.141201
2,-1.167245,0.016736,0.148026
3,-1.124008,0.023188,0.098259
4,-1.081750,0.018176,0.100763
...,...,...,...
466,1.913976,-0.446665,-0.276386
467,1.972936,-0.547597,-0.218687
468,1.860384,-0.608795,-0.265820
469,1.878246,-0.587851,-0.291622


In [264]:
#### plotting factor scores for the entire timestep
fig_fcs = go.Figure()

for fcs in factor_scores_df.columns:
    fig_fcs.add_trace(
        go.Scatter(
            x=factor_scores_df.index,
            y=factor_scores_df[fcs],
            mode='markers',
            text=[f"{p:.2f}%" for p in factor_scores_df[fcs]],
            name=fcs,
        )
    )
    
    fig_fcs.update_layout(
        title='Factor Scores plot across time',
        xaxis={'title': 'Maturities', 'showgrid': False},
        yaxis={'title': 'Change in Yields', 'showgrid': False},
        legend={'title': 'Maturities'},
        template='plotly_white',
        width=950,
        height=600,
    )

fig_fcs.show()

#### PCA sklearn approach

In [249]:
scaler = StandardScaler()
scaled_tyield_data = scaler.fit_transform(tyield_df.values)

In [250]:
pca = PCA()
pca.fit(scaled_tyield_data)

PCA()

In [251]:
eigenvectors_yields_sk = pca.components_
explained_variance = pca.explained_variance_ratio_

In [252]:
#### Plot explained PCA
fig_exp_var = go.Figure(
    data = go.Scatter(
        x = np.arange(len(eigenvectors_yields_sk)),
        y = np.cumsum(explained_variance),
        mode = 'lines+markers',
        text=[f"{v:.3f}" for v in np.cumsum(explained_variance)],
        marker=dict(color='teal')
    )
)

fig_exp_var.update_layout(
    title='Explained variance explained plot',
    xaxis_title="Number of Principal Components",
    yaxis_title="Cumulative Variance Explained",
    template='plotly_white',
    width=1000,
    height=700,
)

fig_exp_var.show()

In [253]:
pc_df = pd.DataFrame(pca.components_[:3], columns=tyield_df.columns, index=['PC1', 'PC2', 'PC3'])
pc_df

,1-Month,3-Month,6-Month,1-Year,2-Year,3-Year,5-Year,7-year,10-Year,20-Year,30-Year
PC1,0.292129,0.295347,0.295538,0.301404,0.318593,0.323840,0.319680,0.311140,0.304445,0.289680,0.259354
PC2,-0.342122,-0.338275,-0.341853,-0.307421,-0.144205,-0.006611,0.137909,0.238827,0.294919,0.371816,0.484801
PC3,0.374143,0.308258,0.190158,-0.092065,-0.369433,-0.382955,-0.323783,-0.221209,-0.039643,0.274071,0.454717


In [254]:
fig_pcs_2 = go.Figure()

for pc in pc_df.index:
    fig_pcs_2.add_trace(
        go.Scatter(
            x=pc_df.columns,
            y=pc_df.loc[pc],
            mode='lines + markers',
            line={'dash':'dash'},
            name=str(pc),
        )
    )
    
    fig_pcs_2.update_layout(
        title='Eigenvector Loadings for selected eigenvectors',
        xaxis={'title': 'Maturities'},
        yaxis={'title': 'Loadings'},
        legend={'title': 'Maturities'},
        template='plotly_white',
        width=1000,
        height=700,
    )

fig_pcs_2.show()

##### Factor scores overtime sklearn PCA

In [256]:
y_pca_score = pca.transform(scaled_tyield_data)
y_pca_score

array([[-3.32038604e+00,  3.98088252e-01,  3.67011816e-01, ...,
         2.27373233e-02,  5.21304581e-02, -1.92281025e-02],
       [-3.05607058e+00,  5.20892466e-01,  4.57231366e-01, ...,
        -2.87613804e-02,  5.18383989e-02, -3.45927258e-03],
       [-3.39463401e+00,  2.05614049e-01,  4.81384311e-01, ...,
        -1.12090012e-02,  3.92399519e-02, -1.37703938e-02],
       ...,
       [ 5.21937431e+00, -2.09500418e+00, -7.96324869e-01, ...,
         4.52307429e-03, -2.10875651e-02, -2.11899067e-02],
       [ 5.27835532e+00, -2.04724092e+00, -8.72863831e-01, ...,
         2.05065025e-02, -2.78950848e-02, -4.96293535e-04],
       [ 5.19901748e+00, -2.22047907e+00, -7.19829820e-01, ...,
        -1.70934419e-02, -1.83565798e-02,  9.61146609e-03]])

In [257]:
y_pca_df = pd.DataFrame(y_pca_score[:, :3], index=tyield_df.index, columns=['PC1', 'PC2', 'PC3'])
y_pca_df

,PC1,PC2,PC3
2016-02-16,-3.320386,0.398088,0.367012
2016-02-17,-3.056071,0.520892,0.457231
2016-02-18,-3.394634,0.205614,0.481384
2016-02-19,-3.274945,0.200109,0.325211
2016-02-22,-3.149235,0.185332,0.335099
...,...,...,...
2017-12-22,5.442231,-1.596118,-0.818938
2017-12-26,5.587949,-1.888788,-0.651652
2017-12-27,5.219374,-2.095004,-0.796325
2017-12-28,5.278355,-2.047241,-0.872864


In [262]:
fig_fcs_2 = go.Figure()

for fcs in y_pca_df.columns:
    fig_fcs_2.add_trace(
        go.Scatter(
            x=y_pca_df.index,
            y=y_pca_df[fcs],
            mode='markers',
            #text=[f"{p:.2f}%" for p in factor_scores_df[fcs]],
            name=fcs,
        )
    )
    
    fig_fcs_2.update_layout(
        title='Factor Scores plot across time (sklearn appraoch)',
        xaxis={'title': 'Maturities', 'showgrid': False},
        yaxis={'title': 'Change in Yields', 'showgrid': False},
        legend={'title': 'Factors Scores'},
        template='plotly_white',
        width=950,
        height=600,
    )

fig_fcs_2.show()

In [212]:
factor_scores_df

,PC1,PC2,PC3
2016-02-16,-3.476357,2.690642,0.866704
2016-02-17,-3.559342,2.728937,0.896161
2016-02-18,-3.457143,2.624337,0.902987
2016-02-19,-3.500380,2.630789,0.853220
2016-02-22,-3.542638,2.625777,0.855723
...,...,...,...
2017-12-22,-6.538364,2.160936,0.478574
2017-12-26,-6.597324,2.060004,0.536274
2017-12-27,-6.484772,1.998806,0.489141
2017-12-28,-6.502635,2.019750,0.463339


##### Getting Residuals

In [280]:
# Reduce to k components
X_pca_scores = pca.transform(scaled_tyield_data)[:, :k]
components_k = pca.components_[:k, :]

# Reconstruct the approximation
X_approx = X_pca_scores @ components_k

# Add back the mean, since the data was centered manually, skip if using StandardScaler
#X_approx += scaler.mean_ 

residuals = scaled_tyield_data - X_approx
residuals

array([[-0.03529262,  0.02241503,  0.01689707, ..., -0.02087484,
        -0.07791193,  0.10561865],
       [ 0.0423962 , -0.04191915, -0.00832558, ..., -0.04390683,
        -0.06518294,  0.09148022],
       [ 0.02440029, -0.0560215 ,  0.03549691, ..., -0.02705025,
        -0.08005821,  0.0886933 ],
       ...,
       [-0.16145456,  0.15121989, -0.02245728, ..., -0.00995232,
        -0.0284414 ,  0.05461235],
       [-0.10437843,  0.03601831,  0.01906187, ..., -0.01500326,
        -0.01037247,  0.05096326],
       [ 0.06623286, -0.04632559, -0.07388114, ..., -0.02376948,
        -0.02879257,  0.04719551]])

In [281]:
residuals_df = pd.DataFrame(residuals, index=tyield_df.index, columns=tyield_df.columns)
residuals_df

,1-Month,3-Month,6-Month,1-Year,2-Year,3-Year,5-Year,7-year,10-Year,20-Year,30-Year
2016-02-16,-0.035293,0.022415,0.016897,-0.048271,0.065216,0.065339,-0.009764,-0.077257,-0.020875,-0.077912,0.105619
2016-02-17,0.042396,-0.041919,-0.008326,-0.024718,0.032045,0.073461,0.002121,-0.052218,-0.043907,-0.065183,0.091480
2016-02-18,0.024400,-0.056022,0.035497,-0.017373,0.013099,0.044377,0.021477,-0.041212,-0.027050,-0.080058,0.088693
2016-02-19,-0.012674,-0.017585,0.056008,-0.069518,0.066925,0.033307,0.017513,-0.053360,-0.038031,-0.069881,0.092591
2016-02-22,0.000505,-0.007747,0.011924,-0.053880,0.088576,0.025463,0.010601,-0.057594,-0.041526,-0.071575,0.101399
...,...,...,...,...,...,...,...,...,...,...,...
2017-12-22,-0.135404,-0.041441,0.114593,0.075168,0.068276,-0.030127,-0.032344,-0.052590,-0.045671,0.015135,0.075181
2017-12-26,-0.076730,0.150043,-0.116468,0.013836,0.071538,0.013990,-0.012436,-0.049351,-0.027114,-0.027979,0.064465
2017-12-27,-0.161455,0.151220,-0.022457,0.048212,0.015510,-0.010951,0.002881,-0.034076,-0.009952,-0.028441,0.054612
2017-12-28,-0.104378,0.036018,0.019062,0.066651,0.035510,-0.029869,-0.019308,-0.022440,-0.015003,-0.010372,0.050963


In [285]:
fig_res = go.Figure()

for mat in residuals_df.columns:
    if mat == '2-Year':
        fig_res.add_trace(
            go.Scatter(
                x=residuals_df.index,
                y=residuals_df[mat],
                mode='lines + markers',
                line={'dash':'dash'},
                name=mat,
            )
        )
        
        fig_res.update_layout(
            title='Residuals for Yield Curve Key Rate (2Y) vs. PCA',
            xaxis={'title': 'Maturities', 'showgrid': False},
            yaxis={'title': 'yields', 'showgrid': False},
            template='plotly_white',
            width=1000,
            height=700,
        )

fig_res.show()

Residuals provides useful information, we want to buy (long) April 2016 at the peak points at the above the zero, and sell at the trough (low point) July 2016

##### Comparing PC2 vs US10y-1yr

In [277]:
comp_v1 = pd.DataFrame(tyield_df['10-Year']-tyield_df['1-Year'], columns=['US10y-US1y'])
comp_v1['PC2'] = factor_scores_df['PC2'].values
comp_v1

,US10y-US1y,PC2
2016-02-16,1.27,0.083041
2016-02-17,1.28,0.121336
2016-02-18,1.22,0.016736
2016-02-19,1.23,0.023188
2016-02-22,1.22,0.018176
...,...,...
2017-12-22,0.75,-0.446665
2017-12-26,0.72,-0.547597
2017-12-27,0.67,-0.608795
2017-12-28,0.67,-0.587851


In [279]:
fig_comp_plot = go.Figure()

for fcs in comp_v1.columns:
    fig_comp_plot.add_trace(
        go.Scatter(
            x=comp_v1.index,
            y=comp_v1[fcs],
            mode='markers',
            #text=[f"{p:.2f}%" for p in factor_scores_df[fcs]],
            name=fcs,
        )
    )
    
    fig_comp_plot.update_layout(
        title='Factor Scores (PC2) vs US10y-US1y',
        xaxis={'title': 'Maturities', 'showgrid': False},
        yaxis={'title': 'Change in yields(PC2), yield diff. (US10y-us1y)', 'showgrid': False},
        template='plotly_white',
        width=800,
        height=600,
    )

fig_comp_plot.show()